# AC-MOT Replay Optimization — V1

**Goal:** improve AC-MOT quickly from the existing detection cache, without rerunning YOLO.

V1 does a focused ByteTrack identity-stability sweep around the current tuned tracker.

It will:
- mount Google Drive;
- clone/pull `AhmedCode110/ACMOT-Codex-V10Style-Portable`;
- locate and verify `detection_cache_v10_p3_yolov8n_fp32_640_736_832`;
- use the repaired VisDrone GT annotations only (no image staging);
- replay the current SCI + resolution policy using cached 640/736/832 detections;
- test 7 small tracker variants;
- calculate combined MOTA / IDF1 / IDS;
- run official TrackEval HOTA/CLEAR/Identity only on the current config + best preliminary challengers;
- save all predictions, per-sequence metrics, official metrics, and the V1 winner to Drive;
- never overwrite the frozen FP16 v10_p4 final result.

**Scientific mode:** `FP32_REPLAY_DEVELOPMENT`.

Replay is for development/candidate selection. It is **not** live FP16 FPS evidence.

Every fifth notebook version (V5, V10, V15, ...) will automatically compare the latest five saved version winners when they are scientifically compatible.

In [ ]:
# CELL 1 — MOUNT DRIVE, CLONE/PULL REPO, INSTALL REPLAY DEPENDENCIES
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import subprocess, sys, os, json, gzip, hashlib, time, math, shutil
from datetime import datetime

REPO = Path("/content/ACMOT-Codex-V10Style-Portable")
if not REPO.exists():
    subprocess.run([
        "git", "clone",
        "https://github.com/AhmedCode110/ACMOT-Codex-V10Style-Portable.git",
        str(REPO)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull"], check=True)

os.chdir(REPO)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "ultralytics==8.3.200",
    "motmetrics",
    "pandas",
    "numpy",
    "scipy",
    "lap",
    "pyyaml",
    "tqdm"
], check=True)

VERSION = 1
VERSION_NAME = f"V{VERSION}"
MODE_LABEL = "FP32_REPLAY_DEVELOPMENT"

DRIVE_ROOT = Path("/content/drive/MyDrive/VisDrone_Results")
VERSION_ROOT = DRIVE_ROOT / "ACMOT_REPLAY_VERSIONS"
OUT_DIR = VERSION_ROOT / VERSION_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True
).strip()

print("Repo commit:", REPO_COMMIT)
print("VERSION =", VERSION_NAME)
print("OUTPUT =", OUT_DIR)
print("Scientific mode =", MODE_LABEL)
print("YOLO inference = DISABLED")

In [ ]:
# CELL 2 — LOCATE + VERIFY LEGACY DETECTION CACHE AND GT
CACHE_NAME = "detection_cache_v10_p3_yolov8n_fp32_640_736_832"

preferred_cache = (
    Path("/content/drive/MyDrive/VisDrone_Results/ACMOT_CODEX_V10STYLE")
    / CACHE_NAME
)

cache_candidates = []
if preferred_cache.exists():
    cache_candidates.append(preferred_cache)

for p in Path("/content/drive").rglob(CACHE_NAME):
    if p.is_dir() and p not in cache_candidates:
        cache_candidates.append(p)

if not cache_candidates:
    raise RuntimeError(
        "Existing FP32 replay cache was not found. "
        "Do NOT run YOLO. Make the shared legacy cache visible to this Colab account first."
    )

CACHE_DIR = cache_candidates[0]
meta_path = CACHE_DIR / "cache_meta.json"

if not meta_path.exists():
    raise RuntimeError("cache_meta.json missing. Refusing unverified cache.")

cache_meta = json.loads(meta_path.read_text(encoding="utf-8"))

if sorted(int(x) for x in cache_meta.get("sizes", [])) != [640, 736, 832]:
    raise RuntimeError("Cache does not contain the required 640/736/832 detection banks.")

print("Using cache:", CACHE_DIR)
print(json.dumps(cache_meta, indent=2))

# Find repaired GT annotations. Replay does not need images.
ann_candidates = [
    Path("/content/drive/MyDrive/visdrone/VisDrone_Zips/VisDrone2019-MOT-test-dev/VisDrone2019-MOT-test-dev/annotations")
]
for p in Path("/content/drive").rglob("VisDrone2019-MOT-test-dev"):
    a = p / "annotations"
    if a.is_dir() and a not in ann_candidates:
        ann_candidates.append(a)

ANN_DIR = next((p for p in ann_candidates if p.exists()), None)
if ANN_DIR is None:
    raise RuntimeError("VisDrone GT annotations not found.")

seqs = sorted(p.stem for p in ANN_DIR.glob("*.txt"))
if len(seqs) != 17:
    raise RuntimeError(f"Expected 17 GT annotation files, found {len(seqs)}.")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

verification = []
for seq in seqs:
    data_file = CACHE_DIR / f"{seq}.jsonl.gz"
    done_file = CACHE_DIR / f"{seq}.complete.json"

    if not data_file.exists() or not done_file.exists():
        raise RuntimeError(f"Missing cache file/receipt for {seq}")

    receipt = json.loads(done_file.read_text(encoding="utf-8"))
    digest = sha256_file(data_file)
    valid = digest == receipt.get("sha256")

    if not valid:
        raise RuntimeError(f"SHA256 mismatch for {seq}")

    verification.append({
        "sequence": seq,
        "frames": int(receipt["frames"]),
        "sha256_ok": True,
    })

total_frames = sum(x["frames"] for x in verification)

print(f"Verified cache: {len(verification)}/17 sequences")
print("Total cached frames:", total_frames)
print("GT:", ANN_DIR)
print("YOLO inference required: NO")
print("Image staging required: NO")

(OUT_DIR / "cache_verification.json").write_text(
    json.dumps({
        "version": VERSION_NAME,
        "scientific_mode": MODE_LABEL,
        "cache_dir": str(CACHE_DIR),
        "cache_meta": cache_meta,
        "gt_dir": str(ANN_DIR),
        "sequences": verification,
        "total_frames": total_frames,
        "verified_at": datetime.now().isoformat(),
    }, indent=2),
    encoding="utf-8",
)

In [ ]:
# CELL 3 — REPLAY CORE (CACHED DETECTIONS -> SCI -> BYTETRACK)
import numpy as np
import pandas as pd
import motmetrics as mm
from collections import deque
from types import SimpleNamespace
from dataclasses import dataclass
from tqdm.auto import tqdm
from ultralytics.trackers.byte_tracker import BYTETracker
from ultralytics.engine.results import Boxes

VISDRONE_GT_CLASSES = [1, 4, 5, 6, 9]

@dataclass
class SceneState:
    sci: float = 0.0
    scene: str = "clear"
    brightness: float = 128.0
    blur: float = 500.0
    edge_density: float = 0.0
    crowd: float = 0.0
    tiny_ratio: float = 0.0
    n_dets: int = 0

class CachedSceneAnalyzer:
    def __init__(self, window=7):
        self.hist = deque(maxlen=int(window))

def load_gt(path):
    cols = ["frame","id","x","y","w","h","score","cat","trunc","occ"]
    df = pd.read_csv(path, header=None, names=cols)
    df = df[df["cat"].isin(VISDRONE_GT_CLASSES)]
    df = df[(df["score"] == 1) & (df["occ"] < 2) & (df["trunc"] < 2)]
    return df.reset_index(drop=True)

def iou_distance(pred_xyxy, gt_xyxy, max_distance=0.5):
    pred = np.asarray(pred_xyxy, dtype=float).reshape(-1, 4)
    gt = np.asarray(gt_xyxy, dtype=float).reshape(-1, 4)

    if not len(pred) or not len(gt):
        return np.empty((len(gt), len(pred)))

    ix1 = np.maximum(pred[:, 0][None, :], gt[:, 0][:, None])
    iy1 = np.maximum(pred[:, 1][None, :], gt[:, 1][:, None])
    ix2 = np.minimum(pred[:, 2][None, :], gt[:, 2][:, None])
    iy2 = np.minimum(pred[:, 3][None, :], gt[:, 3][:, None])

    inter = np.maximum(0, ix2 - ix1) * np.maximum(0, iy2 - iy1)

    ap = (pred[:, 2] - pred[:, 0]) * (pred[:, 3] - pred[:, 1])
    ag = (gt[:, 2] - gt[:, 0]) * (gt[:, 3] - gt[:, 1])
    union = ap[None, :] + ag[:, None] - inter

    iou = np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)
    dist = 1.0 - iou

    # Standard 0.5 IoU matching gate for quick motmetrics screening.
    dist[dist > float(max_distance)] = np.nan
    return dist

def make_tracker(tp):
    return BYTETracker(SimpleNamespace(
        track_high_thresh=float(tp["high"]),
        track_low_thresh=float(tp["low"]),
        new_track_thresh=float(tp["new"]),
        track_buffer=int(tp["buffer"]),
        match_thresh=float(tp["match"]),
        fuse_score=bool(tp.get("fuse", True)),
    ), frame_rate=30)

def analyze_cached_visual(analyzer, visual, prev_boxes):
    brightness = float(visual.get("brightness", 128.0))
    blur = float(visual.get("blur", 500.0))
    edge_density = float(visual.get("edge_density", 0.0))

    n = len(prev_boxes)
    crowd = min(n / 30.0, 1.0)

    if n:
        areas = (
            (prev_boxes[:, 2] - prev_boxes[:, 0])
            * (prev_boxes[:, 3] - prev_boxes[:, 1])
        )
        tiny_ratio = float(np.mean(areas < 32 * 32))
    else:
        tiny_ratio = 0.0

    raw = (
        0.30 * crowd
        + 0.30 * tiny_ratio
        + 0.20 * min(edge_density / 0.14, 1.0)
        + 0.10 * float(brightness < 80)
        + 0.05 * float(blur < 180)
    )

    analyzer.hist.append(float(np.clip(raw, 0.0, 1.0)))
    sci = float(np.mean(analyzer.hist))

    if brightness < 80:
        scene = "night"
    elif blur < 180:
        scene = "blur"
    elif tiny_ratio > 0.50:
        scene = "tiny"
    elif crowd > 0.65 or edge_density > 0.13:
        scene = "crowded"
    else:
        scene = "clear"

    return SceneState(
        sci=sci,
        scene=scene,
        brightness=brightness,
        blur=blur,
        edge_density=edge_density,
        crowd=crowd,
        tiny_ratio=tiny_ratio,
        n_dets=n,
    )

def adaptive_params(state):
    # V1 keeps the current AC-MOT SCI/calibrator policy fixed.
    # The legacy cache was generated with detector NMS IoU=0.45,
    # so V1 does not claim to sweep NMS IoU from cache.
    conf = 0.245 - 0.050 * state.sci
    if state.scene in {"crowded", "tiny", "night"}:
        conf -= 0.012
    conf = float(np.clip(conf, 0.19, 0.28))

    if state.sci > 0.60 or state.tiny_ratio > 0.50:
        imgsz = 832
    elif state.sci > 0.35 or state.scene in {"crowded", "tiny"}:
        imgsz = 736
    else:
        imgsz = 640

    return {"conf": conf, "imgsz": imgsz}

def track_from_cache(tracker, dets, shape, conf, tp):
    arr = (
        np.asarray(dets, dtype=float).reshape(-1, 6)
        if len(dets)
        else np.empty((0, 6), dtype=float)
    )

    if len(arr):
        arr = arr[arr[:, 4] >= float(conf)]

    tracker.args.track_high_thresh = float(tp["high"])
    tracker.args.track_low_thresh = float(tp["low"])
    tracker.args.new_track_thresh = float(tp["new"])
    tracker.args.match_thresh = float(tp["match"])

    out = np.asarray(
        tracker.update(Boxes(arr, tuple(shape))),
        dtype=float
    ).reshape(-1, 8)

    if len(out):
        ids = out[:, 4].astype(int)
        boxes = out[:, :4]
        scores = out[:, 5]
    else:
        ids = np.array([], dtype=int)
        boxes = np.empty((0, 4), dtype=float)
        scores = np.array([], dtype=float)

    return ids, boxes, scores

In [ ]:
# CELL 4 — V1 DRY RUN: SMALL TRACKER SWEEP
CURRENT_TRACKER = {
    "high": 0.18,
    "low": 0.04,
    "new": 0.20,
    "buffer": 45,
    "match": 0.86,
    "fuse": True,
}

TRIALS = [
    {"name":"V1_CURRENT", **CURRENT_TRACKER},
    {"name":"V1_MATCH_084", **(CURRENT_TRACKER | {"match":0.84})},
    {"name":"V1_MATCH_088", **(CURRENT_TRACKER | {"match":0.88})},
    {"name":"V1_NEW22_MATCH88", **(CURRENT_TRACKER | {"new":0.22, "match":0.88})},
    {"name":"V1_NEW24_MATCH88", **(CURRENT_TRACKER | {"new":0.24, "match":0.88})},
    {"name":"V1_BUFFER60", **(CURRENT_TRACKER | {"buffer":60})},
    {"name":"V1_BUFFER60_NEW22_MATCH88", **(CURRENT_TRACKER | {"buffer":60, "new":0.22, "match":0.88})},
]

print("V1 DRY RUN")
print("Candidates:", len(TRIALS))
print("Sequences:", len(seqs))
print("Cached frames:", total_frames)
print("Cache:", CACHE_DIR)
print("Required detection banks: 640, 736, 832")
print("YOLO inference required: NO")
print("GPU inference required: NO")
print("Image staging required: NO")
print("Will run: cache -> current SCI policy -> ByteTrack -> quick motmetrics")
print("")
for t in TRIALS:
    print(t["name"], t)

In [ ]:
# CELL 5 — RUN THE V1 REPLAY SWEEP
MOT_METRICS = [
    "mota", "idf1", "num_switches", "recall", "precision",
    "num_misses", "num_false_positives", "num_matches"
]

def run_trial(trial):
    rows = []
    accs = []
    acc_names = []

    pred_dir = OUT_DIR / "predictions" / trial["name"]
    pred_dir.mkdir(parents=True, exist_ok=True)

    pbar = tqdm(total=total_frames, desc=trial["name"], dynamic_ncols=True)

    for seq in seqs:
        gt = load_gt(ANN_DIR / f"{seq}.txt")
        tracker = make_tracker(trial)
        analyzer = CachedSceneAnalyzer(window=7)
        state = SceneState()
        prev_boxes = np.empty((0, 4), dtype=float)
        acc = mm.MOTAccumulator(auto_id=True)

        pred_lines = []
        sizes = []
        confs = []
        replay_times = []

        cache_file = CACHE_DIR / f"{seq}.jsonl.gz"

        with gzip.open(cache_file, "rt", encoding="utf-8") as f:
            for line in f:
                t0 = time.perf_counter()
                rec = json.loads(line)
                frame_id = int(rec["frame"])

                if frame_id == 1 or frame_id % 10 == 1:
                    state = analyze_cached_visual(
                        analyzer,
                        rec.get("visual", {}),
                        prev_boxes
                    )

                params = adaptive_params(state)
                cached_dets = rec["bank"][str(params["imgsz"])]

                ids, boxes, scores = track_from_cache(
                    tracker,
                    cached_dets,
                    rec["shape"],
                    params["conf"],
                    trial
                )

                prev_boxes = boxes.copy()

                for tid, box, score in zip(ids, boxes, scores):
                    x1, y1, x2, y2 = box.tolist()
                    pred_lines.append(
                        f"{frame_id},{int(tid)},{x1:.2f},{y1:.2f},"
                        f"{x2-x1:.2f},{y2-y1:.2f},{float(score):.6f},-1,-1,-1\n"
                    )

                gt_f = gt[gt["frame"] == frame_id]
                gt_ids = gt_f["id"].to_numpy()

                if len(gt_f):
                    gt_boxes = np.column_stack([
                        gt_f["x"],
                        gt_f["y"],
                        gt_f["x"] + gt_f["w"],
                        gt_f["y"] + gt_f["h"],
                    ])
                else:
                    gt_boxes = np.empty((0, 4), dtype=float)

                dist = iou_distance(boxes, gt_boxes, max_distance=0.5)
                acc.update(
                    gt_ids,
                    ids,
                    dist if dist.size else np.empty((len(gt_ids), len(ids)))
                )

                sizes.append(params["imgsz"])
                confs.append(params["conf"])
                replay_times.append(time.perf_counter() - t0)
                pbar.update(1)

        (pred_dir / f"{seq}.txt").write_text(
            "".join(pred_lines),
            encoding="utf-8"
        )

        mh = mm.metrics.create()
        seq_result = mh.compute(acc, metrics=MOT_METRICS, name=seq).iloc[0]

        rows.append({
            "version": VERSION_NAME,
            "mode": MODE_LABEL,
            "trial": trial["name"],
            "sequence": seq,
            "frames": len(sizes),
            "replay_fps_not_live": len(sizes) / max(sum(replay_times), 1e-9),
            "mean_imgsz": float(np.mean(sizes)),
            "mean_conf": float(np.mean(confs)),
            "mota": float(seq_result["mota"]),
            "idf1": float(seq_result["idf1"]),
            "ids": int(seq_result["num_switches"]),
            "recall": float(seq_result["recall"]),
            "precision": float(seq_result["precision"]),
            "fn": int(seq_result["num_misses"]),
            "fp": int(seq_result["num_false_positives"]),
            "matches": int(seq_result["num_matches"]),
        })

        accs.append(acc)
        acc_names.append(seq)

    pbar.close()

    # Proper combined motmetrics screening row across all 17 sequences.
    mh = mm.metrics.create()
    combined_table = mh.compute_many(
        accs,
        names=acc_names,
        metrics=MOT_METRICS,
        generate_overall=True
    )
    r = combined_table.loc["OVERALL"]

    overall = {
        "version": VERSION_NAME,
        "mode": MODE_LABEL,
        "trial": trial["name"],
        "mota": float(r["mota"]),
        "idf1": float(r["idf1"]),
        "ids": int(r["num_switches"]),
        "recall": float(r["recall"]),
        "precision": float(r["precision"]),
        "fn": int(r["num_misses"]),
        "fp": int(r["num_false_positives"]),
        "matches": int(r["num_matches"]),
        "mean_imgsz": float(pd.DataFrame(rows)["mean_imgsz"].mean()),
        "replay_fps_not_live": float(pd.DataFrame(rows)["replay_fps_not_live"].mean()),
    }

    return pd.DataFrame(rows), overall

all_rows = []
overall_rows = []

for trial in TRIALS:
    print("\nRunning:", trial["name"])
    per_seq, overall = run_trial(trial)
    per_seq.to_csv(OUT_DIR / f"{trial['name']}_per_sequence.csv", index=False)
    all_rows.append(per_seq)
    overall_rows.append(overall)

replay_df = pd.concat(all_rows, ignore_index=True)
replay_df.to_csv(OUT_DIR / "V1_all_per_sequence.csv", index=False)

prelim = pd.DataFrame(overall_rows)
prelim.to_csv(OUT_DIR / "V1_preliminary_combined.csv", index=False)

print("\nPRELIMINARY COMBINED METRICS")
display(prelim)
print("YOLO was not executed.")

In [ ]:
# CELL 6 — PRELIMINARY RANKING AND TRACKEVAL FINALIST SELECTION
current = prelim[prelim["trial"] == "V1_CURRENT"].iloc[0]

prelim["delta_mota_vs_current"] = prelim["mota"] - current["mota"]
prelim["delta_idf1_vs_current"] = prelim["idf1"] - current["idf1"]
prelim["delta_ids_vs_current"] = prelim["ids"] - current["ids"]

def minmax(series, higher_better=True):
    s = series.astype(float)
    lo, hi = float(s.min()), float(s.max())
    if abs(hi - lo) < 1e-12:
        return pd.Series(np.ones(len(s)), index=s.index)
    z = (s - lo) / (hi - lo)
    return z if higher_better else 1.0 - z

prelim["preliminary_balanced_score"] = (
    0.45 * minmax(prelim["idf1"], True)
    + 0.35 * minmax(prelim["mota"], True)
    + 0.20 * minmax(prelim["ids"], False)
)

prelim = prelim.sort_values(
    ["preliminary_balanced_score", "idf1", "mota", "ids"],
    ascending=[False, False, False, True]
).reset_index(drop=True)

prelim.insert(0, "prelim_rank", np.arange(1, len(prelim) + 1))
prelim.to_csv(OUT_DIR / "V1_PRELIMINARY_LEADERBOARD.csv", index=False)

# Official TrackEval: always current + best 3 challengers.
challengers = [
    x for x in prelim["trial"].tolist()
    if x != "V1_CURRENT"
][:3]
TRACKEVAL_FINALISTS = ["V1_CURRENT", *challengers]

print("PRELIMINARY LEADERBOARD")
display(prelim)
print("\nOfficial TrackEval finalists:")
for x in TRACKEVAL_FINALISTS:
    print(" -", x)

print("\nStill no YOLO inference.")

In [ ]:
# CELL 7 — OFFICIAL TRACKEVAL FOR CURRENT + TOP CHALLENGERS (NO YOLO)
TRACK_ROOT = Path("/content/TrackEval")

if not (TRACK_ROOT / "trackeval" / "__init__.py").exists():
    if TRACK_ROOT.exists():
        shutil.rmtree(TRACK_ROOT)
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/JonathonLuiten/TrackEval.git",
        str(TRACK_ROOT)
    ], check=True)

# Compatibility-only patch for modern NumPy.
for py in TRACK_ROOT.rglob("*.py"):
    txt = py.read_text(encoding="utf-8")
    new = (
        txt.replace("np.float", "float")
           .replace("np.int", "int")
           .replace("np.bool", "bool")
    )
    if new != txt:
        py.write_text(new, encoding="utf-8")

if str(TRACK_ROOT) not in sys.path:
    sys.path.insert(0, str(TRACK_ROOT))

import trackeval

GT_PARENT = OUT_DIR / "trackeval_gt"
TR_PARENT = OUT_DIR / "trackeval_trackers"
GT_BENCH = GT_PARENT / "VisDroneACMOT-test"
TR_BENCH = TR_PARENT / "VisDroneACMOT-test"
SEQMAP = OUT_DIR / "seqmap.txt"

SEQMAP.write_text("name\n" + "\n".join(seqs) + "\n", encoding="utf-8")

# Build filtered GT layout.
for seq in seqs:
    dst = GT_BENCH / seq / "gt"
    dst.mkdir(parents=True, exist_ok=True)

    gt = load_gt(ANN_DIR / f"{seq}.txt")

    mot_gt = pd.DataFrame({
        0: gt["frame"].astype(int),
        1: gt["id"].astype(int),
        2: gt["x"],
        3: gt["y"],
        4: gt["w"],
        5: gt["h"],
        6: 1,
        7: 1,
        8: 1,
    })
    mot_gt.to_csv(dst / "gt.txt", header=False, index=False)

# Copy only finalist predictions into TrackEval layout.
for trial in TRACKEVAL_FINALISTS:
    data_dir = TR_BENCH / trial / "data"
    data_dir.mkdir(parents=True, exist_ok=True)
    src_dir = OUT_DIR / "predictions" / trial

    for seq in seqs:
        shutil.copyfile(
            src_dir / f"{seq}.txt",
            data_dir / f"{seq}.txt"
        )

eval_config = trackeval.Evaluator.get_default_eval_config()
eval_config.update({
    "USE_PARALLEL": False,
    "PRINT_RESULTS": True,
    "PRINT_ONLY_COMBINED": True,
    "PRINT_CONFIG": False,
    "OUTPUT_SUMMARY": True,
    "OUTPUT_DETAILED": True,
    "PLOT_CURVES": False,
})

dataset_config = trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
dataset_config.update({
    "GT_FOLDER": str(GT_PARENT),
    "TRACKERS_FOLDER": str(TR_PARENT),
    "TRACKERS_TO_EVAL": TRACKEVAL_FINALISTS,
    "BENCHMARK": "VisDroneACMOT",
    "SPLIT_TO_EVAL": "test",
    "SEQMAP_FILE": str(SEQMAP),
    "DO_PREPROC": False,
    "TRACKER_SUB_FOLDER": "data",
    "OUTPUT_SUB_FOLDER": "",
    "PRINT_CONFIG": False,
})

metrics_config = {
    "METRICS": ["HOTA", "CLEAR", "Identity"],
    "THRESHOLD": 0.5
}

print("Running official TrackEval from cached replay predictions.")
print("YOLO inference: NO")

evaluator = trackeval.Evaluator(eval_config)
dataset_list = [trackeval.datasets.MotChallenge2DBox(dataset_config)]
metrics_list = [
    trackeval.metrics.HOTA(metrics_config),
    trackeval.metrics.CLEAR(metrics_config),
    trackeval.metrics.Identity(metrics_config),
]

_ = evaluator.evaluate(dataset_list, metrics_list)

# Parse generated pedestrian_summary.txt files.
official_rows = []

for trial in TRACKEVAL_FINALISTS:
    candidates = list((TR_BENCH / trial).rglob("pedestrian_summary.txt"))
    if not candidates:
        raise RuntimeError(f"TrackEval summary missing for {trial}")

    summary_file = candidates[0]
    sdf = pd.read_csv(summary_file, sep=r"\s+")
    row = sdf.iloc[0]

    official_rows.append({
        "version": VERSION_NAME,
        "trial": trial,
        "hota": float(row["HOTA"]),
        "mota": float(row["MOTA"]),
        "idf1": float(row["IDF1"]),
        "ids": int(row["IDSW"]),
        "summary_file": str(summary_file),
    })

official = pd.DataFrame(official_rows)
official.to_csv(OUT_DIR / "V1_OFFICIAL_TRACKEVAL.csv", index=False)

print("\nOFFICIAL TRACKEVAL V1 FINALISTS")
display(official)

In [ ]:
# CELL 8 — FINAL V1 LEADERBOARD + WINNER
official_current = official[official["trial"] == "V1_CURRENT"].iloc[0]

official["delta_hota_vs_current"] = official["hota"] - official_current["hota"]
official["delta_mota_vs_current"] = official["mota"] - official_current["mota"]
official["delta_idf1_vs_current"] = official["idf1"] - official_current["idf1"]
official["delta_ids_vs_current"] = official["ids"] - official_current["ids"]

official["balanced_score"] = (
    0.40 * minmax(official["hota"], True)
    + 0.35 * minmax(official["idf1"], True)
    + 0.15 * minmax(official["mota"], True)
    + 0.10 * minmax(official["ids"], False)
)

official = official.sort_values(
    ["balanced_score", "hota", "idf1", "mota", "ids"],
    ascending=[False, False, False, False, True]
).reset_index(drop=True)

official.insert(0, "rank", np.arange(1, len(official) + 1))
official.to_csv(OUT_DIR / "LEADERBOARD_V1.csv", index=False)

best = official.iloc[0].to_dict()

print("FINAL V1 LEADERBOARD — OFFICIAL TRACKEVAL")
display(official)

print("\nV1 BEST CANDIDATE:")
print(json.dumps(best, indent=2, default=float))

if best["trial"] == "V1_CURRENT":
    print("\nV1 did not beat the current replay tracker. V2 should explore another local region.")
else:
    print("\nV1 found a replay candidate better than the current replay reference.")

print("\nThis still does NOT replace the frozen live FP16 result.")

In [ ]:
# CELL 9 — SAVE V1 RESULT + UPDATE VERSION HISTORY
result_payload = {
    "version": VERSION_NAME,
    "version_number": VERSION,
    "status": "COMPLETE",
    "scientific_mode": MODE_LABEL,
    "focus": "ByteTrack identity-stability local sweep",
    "cache_dir": str(CACHE_DIR),
    "cache_meta": cache_meta,
    "dataset_gt": str(ANN_DIR),
    "sequences": len(seqs),
    "frames": total_frames,
    "repo_commit": REPO_COMMIT,
    "created_at": datetime.now().isoformat(),
    "current_replay_reference_official": official_current.to_dict(),
    "best_candidate_official": best,
    "trackeval_finalists": TRACKEVAL_FINALISTS,
    "trials": TRIALS,
    "frozen_live_fp16_reference": {
        "HOTA": 22.856,
        "MOTA": 11.607,
        "IDF1": 21.963,
        "IDS": 270,
        "FPS": 25.99,
        "warning": "Different precision/evidence mode. Do not directly merge live FP16 and FP32 replay metrics."
    }
}

(OUT_DIR / "VERSION_RESULT.json").write_text(
    json.dumps(result_payload, indent=2, default=float),
    encoding="utf-8"
)

(OUT_DIR / "VERSION_COMPLETE.txt").write_text(
    f"{VERSION_NAME} complete from cached detections. No YOLO inference.\n",
    encoding="utf-8"
)

history_rows = []

for d in sorted(VERSION_ROOT.glob("V*")):
    rp = d / "VERSION_RESULT.json"
    if not rp.exists():
        continue

    try:
        r = json.loads(rp.read_text(encoding="utf-8"))
        b = r.get("best_candidate_official", {})
        history_rows.append({
            "version": r.get("version"),
            "focus": r.get("focus"),
            "mode": r.get("scientific_mode"),
            "best_trial": b.get("trial"),
            "hota": b.get("hota"),
            "mota": b.get("mota"),
            "idf1": b.get("idf1"),
            "ids": b.get("ids"),
            "balanced_score": b.get("balanced_score"),
            "repo_commit": r.get("repo_commit"),
        })
    except Exception as e:
        print("Skipping unreadable version:", rp, e)

history = pd.DataFrame(history_rows)
if len(history):
    history.to_csv(VERSION_ROOT / "ALL_VERSION_HISTORY.csv", index=False)
    print("ALL VERSION HISTORY")
    display(history)

print("Saved:", OUT_DIR / "VERSION_RESULT.json")
print("Saved:", VERSION_ROOT / "ALL_VERSION_HISTORY.csv")

In [ ]:
# CELL 10 — AUTOMATIC COMPARISON EVERY 5 VERSIONS
if VERSION % 5 != 0:
    print(f"{VERSION_NAME}: automatic 5-version comparison is not due yet.")
    print("It will run automatically at V5, V10, V15, ...")
else:
    start_v = VERSION - 4
    block_rows = []

    for v in range(start_v, VERSION + 1):
        rp = VERSION_ROOT / f"V{v}" / "VERSION_RESULT.json"

        if not rp.exists():
            print("Missing:", rp)
            continue

        r = json.loads(rp.read_text(encoding="utf-8"))
        b = r["best_candidate_official"]

        block_rows.append({
            "version": r["version"],
            "focus": r.get("focus"),
            "mode": r.get("scientific_mode"),
            "trial": b.get("trial"),
            "hota": b.get("hota"),
            "mota": b.get("mota"),
            "idf1": b.get("idf1"),
            "ids": b.get("ids"),
            "balanced_score": b.get("balanced_score"),
        })

    comp = pd.DataFrame(block_rows)

    if len(comp) != 5:
        print(f"Comparison incomplete: {len(comp)}/5 results found.")
    elif comp["mode"].nunique() != 1:
        print("WARNING: scientific modes differ. Direct ranking is blocked.")
    else:
        comp = comp.sort_values(
            ["balanced_score", "hota", "idf1", "mota", "ids"],
            ascending=[False, False, False, False, True]
        ).reset_index(drop=True)

        comp.insert(0, "block_rank", np.arange(1, len(comp) + 1))

        block_name = f"V{start_v}_to_V{VERSION}"
        out_csv = VERSION_ROOT / f"COMPARISON_{block_name}.csv"
        comp.to_csv(out_csv, index=False)

        print("5-VERSION OFFICIAL COMPARISON")
        display(comp)
        print("Best version:", comp.iloc[0]["version"])
        print("Saved:", out_csv)

## Versioning rule from here

- `V1`: tracker identity-stability local sweep.
- `V2`: start from the V1 winner and refine the strongest tracker dimensions or move to one SCI family.
- `V3`, `V4`, ...: one controlled experiment family per notebook.
- `V5`: automatically compare V1–V5 winners.
- `V10`: automatically compare V6–V10, and so on.

Each version saves its own immutable result folder under:

`MyDrive/VisDrone_Results/ACMOT_REPLAY_VERSIONS/V<number>/`

Only the strongest replay candidates should later receive a live FP16 validation.